In [1]:
import sys
import os

# Thêm thư mục src vào hệ thống để import
sys.path.append(os.path.abspath("../src"))

from inventory_simulation import inventory_simulation_model

In [2]:
import pandas as pd
import numpy as np

# Danh sách các mã sản phẩm nhóm B
items = ['B_PRODUCT_01', 'B_PRODUCT_02', 'B_PRODUCT_03']
days = 30 # Số ngày mô phỏng

data_list = []

for item in items:
    # Giả lập nhu cầu khác nhau cho từng mã
    if item == 'B_PRODUCT_01':
        forecast = [15, 12, 18, 14, 15, 16, 13] * 4 + [15, 12]
        actual = [14, 13, 20, 10, 18, 15, 12] * 4 + [16, 11]
    elif item == 'B_PRODUCT_02':
        forecast = [5, 5, 8, 5, 6, 7, 5] * 4 + [5, 6] # Nhu cầu thấp
        actual = [4, 6, 9, 3, 7, 8, 4] * 4 + [6, 5]
    else:
        forecast = [50, 45, 55, 48, 52, 50, 49] * 4 + [50, 51] # Nhu cầu cao
        actual = [55, 40, 60, 45, 55, 52, 48] * 4 + [49, 53]

    for day in range(days):
        data_list.append({
            'item_id': item,
            'day': day + 1,
            'forecast_demand': forecast[day],
            'actual_demand': actual[day]
        })

df_B = pd.DataFrame(data_list)


In [4]:
# CHỈ CẦN THAY MÃ Ở ĐÂY
TARGET_ID = 'B_PRODUCT_03' 

# Lọc dữ liệu cho mã đã chọn
df_subset = df_B[df_B['item_id'] == TARGET_ID].reset_index(drop=True)

# Chạy mô phỏng
df_daily, df_metrics = inventory_simulation_model(df_subset, review_length=3, service_level=0.65)

# --- BƯỚC ĐỔI VỀ SỐ NGUYÊN ---
# 1. Làm tròn và chuyển các cột số trong bảng chi tiết về kiểu int
cols_to_fix = ['Tồn đầu', 'Nhu cầu thực tế', 'Số lượng đặt', 'Tồn cuối']
# Lưu ý: Kiểm tra tên cột trong df_daily của bạn có đúng như trên không
for col in df_daily.columns:
    if col in cols_to_fix or df_daily[col].dtype == 'float64':
        df_daily[col] = df_daily[col].round(0).astype(int)

# 2. Lấy giá trị S mục tiêu dưới dạng số nguyên
target_S_int = int(round(df_metrics['target_level_S'].iloc[0]))

# --- HIỂN THỊ KẾT QUẢ ---
print(f"--- KẾT QUẢ CHO MÃ: {TARGET_ID} (ĐÃ LÀM TRÒN) ---")
print(f"S mục tiêu: {target_S_int}")
print(f"Service Level thực tế (Alpha): {df_metrics['service_level_alpha'].iloc[0]:.2%}")
print(f"Fill Rate thực tế (Beta): {df_metrics['fill_rate_beta'].iloc[0]:.2%}")

# Hiển thị 10 ngày đầu
display(df_daily.head(10))

--- KẾT QUẢ CHO MÃ: B_PRODUCT_03 (ĐÃ LÀM TRÒN) ---
S mục tiêu: 155
Service Level thực tế (Alpha): 83.33%
Fill Rate thực tế (Beta): 99.64%


,Ngày,Tồn đầu,Nhu cầu thực tế,Số lượng đặt,Tồn cuối
0,1,155,55,0,100
1,2,100,40,0,60
2,3,60,60,0,0
3,4,0,45,155,110
4,5,110,55,0,55
5,6,55,52,0,3
6,7,3,48,152,107
7,8,107,55,0,52
8,9,52,40,0,12
9,10,12,60,143,95
